# Phase 1: 베이스라인 실행 — 실패 확인

**목표**: ScriptedBaseline으로 PickCube-v1을 돌리고, 성공/실패율을 확인한다.  
**노선**: Medium-A (scripted policy + 저비용 신호, GPU 있으면 빠름)

성공률 목표 대역: PickCube/StackCube **40~70%**, PickSingleYCB **30~60%**  
범위를 벗어나면 `COLLECT_RAND`의 `obj_xy_std` 조정

In [ ]:
# ── 1. 설치 ─────────────────────────────────────────────
!pip install mani_skill torch numpy pandas pyarrow scikit-learn matplotlib -q
# ManiSkill3 asset 다운로드 (처음 한 번만)
!python -m mani_skill.utils.download_asset 'PickCube-v1' -y

In [ ]:
# ── 2. GitHub 클론 (또는 Drive 마운트) ──────────────────
# 방법 A: GitHub에서 클론
# !git clone https://github.com/YOUR_USERNAME/risk-gated-vla.git
# %cd risk-gated-vla

# 방법 B: Google Drive 마운트
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/risk_gated_vla

import os, sys
# 프로젝트 루트를 PYTHONPATH에 추가
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Working dir:', os.getcwd())

In [ ]:
# ── 3. 환경 확인 ─────────────────────────────────────────
import torch
import mani_skill
print(f'mani_skill : {mani_skill.__version__}')
print(f'torch      : {torch.__version__}')
print(f'CUDA       : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 4. obs 구조 확인 (첫 episode) ───────────────────────
from rgvla.envs.make_env import make_env, RANDOMIZATION_PRESETS

TASK = 'PickCube-v1'
env = make_env(TASK)
env.print_obs_structure(env.reset_with(0, 0)[0])
print('\nAction space:', env.action_space)
print('TCP pos sample:', env.get_tcp_pos(env.reset_with(0, 0)[0]))
try:
    print('Obj pos sample:', env.get_obj_pos(env.reset_with(0, 0)[0]))
    print('Goal pos sample:', env.get_goal_pos(env.reset_with(0, 0)[0]))
except KeyError as e:
    print(f'Key error: {e}  → baseline_scripted.py 의 _get_obj_pos 키 목록 조정 필요')
env.close()

In [ ]:
# ── 5. 베이스라인 실행 ───────────────────────────────────
import time, numpy as np
from rgvla.policy.baseline_scripted import ScriptedBaseline
from rgvla.utils.seeding import make_seed_grid

N_SEEDS    = 5
N_EPS      = 10      # seed당 episode 수
MAX_STEPS  = 200
RAND       = RANDOMIZATION_PRESETS['eval']  # 먼저 eval(낮은 randomization)로 성공률 확인

env    = make_env(TASK, randomization=RAND)
policy = ScriptedBaseline(task=TASK)
grid   = make_seed_grid(N_SEEDS, N_EPS)

results = []
print(f"{'seed':>4} {'ep':>3}  {'ok':>4}  {'steps':>5}  {'phase':>20}  {'ms/step':>9}")
print('─' * 60)

for seed, ep in grid:
    obs, info = env.reset_with(seed, ep)
    policy.reset(obs)
    success, n_steps = False, 0
    t0 = time.perf_counter()

    for step in range(MAX_STEPS):
        action = policy.act(obs)
        obs, _, terminated, truncated, info = env.step(action[np.newaxis, :])
        n_steps = step + 1
        success = env.is_success(info)
        term  = bool(np.any(terminated)) if hasattr(terminated, '__iter__') else bool(terminated)
        trunc = bool(np.any(truncated))  if hasattr(truncated,  '__iter__') else bool(truncated)
        if term or trunc or success:
            break

    ms = (time.perf_counter() - t0) * 1000 / max(n_steps, 1)
    results.append({'seed': seed, 'ep': ep, 'success': success,
                    'n_steps': n_steps, 'ms_per_step': ms, 'phase': policy._phase})
    ok = '✓' if success else '✗'
    print(f"{seed:>4} {ep:>3}  {ok:>4}  {n_steps:>5}  {policy._phase:>20}  {ms:>8.1f}ms")

env.close()

In [ ]:
# ── 6. 결과 집계 + 목표 대역 판정 ───────────────────────
suc = [r['success'] for r in results]
sr  = float(np.mean(suc))

print('═' * 50)
print(f'Episodes  : {len(results)}')
print(f'Success   : {sum(suc)}/{len(results)}  ({sr*100:.1f}%)')
print(f'Mean steps: {np.mean([r["n_steps"] for r in results]):.1f}')
print(f'Latency   : {np.mean([r["ms_per_step"] for r in results]):.2f} ms/step')

lo, hi = 0.40, 0.70
if lo <= sr <= hi:
    print(f'\n✓ 성공률 [{lo*100:.0f}%,{hi*100:.0f}%] 목표 대역 내 — Phase 2 진행 가능')
elif sr > hi:
    print(f'\n△ 너무 높음. RAND["obj_xy_std"] 를 0.10~0.15 로 올려보세요')
else:
    print(f'\n✗ 너무 낮음. RAND["obj_xy_std"] 를 0.03 이하로 낮춰보세요')

In [ ]:
# ── 7. 결정성 테스트 (Phase 1 게이트 1) ─────────────────
import numpy as np

def flatten_obs(obs):
    parts = []
    if isinstance(obs, dict):
        for v in sorted(obs.keys()):
            parts.append(flatten_obs(obs[v]))
    else:
        arr = obs.numpy() if hasattr(obs, 'numpy') else obs
        parts.append(np.asarray(arr, dtype=np.float32).reshape(-1))
    return np.concatenate(parts) if parts else np.array([], dtype=np.float32)

SAMPLE = [(0,0), (0,1), (1,0), (3,5)]
all_pass = True
for seed, ep in SAMPLE:
    e1, e2 = make_env(TASK), make_env(TASK)
    o1, _ = e1.reset_with(seed, ep)
    o2, _ = e2.reset_with(seed, ep)
    diff = np.abs(flatten_obs(o1) - flatten_obs(o2)).max()
    ok   = diff < 1e-5
    all_pass = all_pass and ok
    print(f'  (seed={seed}, ep={ep}) max_diff={diff:.2e}  {"✓" if ok else "✗ FAIL"}')
    e1.close(); e2.close()

if all_pass:
    print('\n✓ 게이트 1 통과: 결정성 확인됨. Phase 2 진행 가능.')
else:
    print('\n✗ 게이트 1 실패: 동일 seed에서 obs가 다름. seeding 로직 점검 필요.')